In [15]:
from pathlib import Path

import numpy as np
import pandas as pd

current_directory = Path.cwd()

if current_directory.name == "notebooks":
    project_root = current_directory.parent
else:
    project_root = current_directory

raw_data_path = (
    project_root
    / "data"
    / "raw"
    / "heart_disease.csv"
)

df_raw = pd.read_csv(raw_data_path)

df_clean = df_raw.copy()

print(f"Raw dataset shape: {df_raw.shape}")
print(f"Working copy shape: {df_clean.shape}")
print(f"Raw file unchanged: {df_raw.equals(df_clean)}")

Raw dataset shape: (4238, 16)
Working copy shape: (4238, 16)
Raw file unchanged: True


In [16]:
column_name_mapping = {
    "Gender": "gender",
    "age": "age",
    "education": "education",
    "currentSmoker": "current_smoker",
    "cigsPerDay": "cigarettes_per_day",
    "BPMeds": "bp_meds",
    "prevalentStroke": "prevalent_stroke",
    "prevalentHyp": "prevalent_hypertension",
    "diabetes": "diabetes",
    "totChol": "total_cholesterol",
    "sysBP": "systolic_bp",
    "diaBP": "diastolic_bp",
    "BMI": "bmi",
    "heartRate": "heart_rate",
    "glucose": "glucose",
    "Heart_ stroke": "heart_disease"
}

missing_columns = [
    column
    for column in column_name_mapping
    if column not in df_clean.columns
]

if missing_columns:
    raise KeyError(
        f"Expected columns were not found: {missing_columns}"
    )

df_clean = df_clean.rename(columns=column_name_mapping)

print("Column names standardised successfully\n")

for index, column in enumerate(df_clean.columns, start=1):
    print(f"{index}. {column}")

Column names standardised successfully

1. gender
2. age
3. education
4. current_smoker
5. cigarettes_per_day
6. bp_meds
7. prevalent_stroke
8. prevalent_hypertension
9. diabetes
10. total_cholesterol
11. systolic_bp
12. diastolic_bp
13. bmi
14. heart_rate
15. glucose
16. heart_disease


In [17]:
clean_target = (
    df_clean["heart_disease"]
    .astype("string")
    .str.strip()
    .str.lower()
)

expected_target_values = {"no", "yes"}
actual_target_values = set(clean_target.dropna().unique())

unexpected_values = actual_target_values - expected_target_values

if unexpected_values:
    raise ValueError(
        f"Unexpected target values found: {unexpected_values}"
    )

df_clean["heart_disease"] = clean_target.map({
    "no": 0,
    "yes": 1
}).astype("int64")

target_check = (
    df_clean["heart_disease"]
    .value_counts()
    .sort_index()
    .rename_axis("Heart Disease")
    .reset_index(name="Count")
)

target_check

,Heart Disease,Count
0,0,3594
1,1,644


In [18]:
df_clean["gender"] = (
    df_clean["gender"]
    .astype("string")
    .str.strip()
    .str.lower()
)

expected_gender_values = {"male", "female"}
actual_gender_values = set(df_clean["gender"].dropna().unique())

unexpected_gender_values = (
    actual_gender_values - expected_gender_values
)

if unexpected_gender_values:
    raise ValueError(
        f"Unexpected gender values found: "
        f"{unexpected_gender_values}"
    )

gender_summary = (
    df_clean["gender"]
    .value_counts(dropna=False)
    .rename_axis("Gender")
    .reset_index(name="Count")
)

gender_summary["Percentage"] = (
    gender_summary["Count"] / len(df_clean) * 100
).round(2)

gender_summary

,Gender,Count,Percentage
0,female,2419,57.08
1,male,1819,42.92


In [19]:
binary_columns = [
    "current_smoker",
    "bp_meds",
    "prevalent_stroke",
    "prevalent_hypertension",
    "diabetes"
]

stroke_values = (
    df_clean["prevalent_stroke"]
    .astype("string")
    .str.strip()
    .str.lower()
)

stroke_mapping = {
    "no": 0,
    "yes": 1,
    "0": 0,
    "1": 1
}

unexpected_stroke_values = (
    set(stroke_values.dropna().unique())
    - set(stroke_mapping.keys())
)

if unexpected_stroke_values:
    raise ValueError(
        f"Unexpected prevalent_stroke values: "
        f"{unexpected_stroke_values}"
    )

df_clean["prevalent_stroke"] = (
    stroke_values
    .map(stroke_mapping)
    .astype("Int64")
)

binary_validation = []

for column in binary_columns:
    df_clean[column] = pd.to_numeric(
        df_clean[column],
        errors="coerce"
    ).astype("Int64")

    unique_values = set(
        df_clean[column].dropna().unique()
    )

    unexpected_values = unique_values - {0, 1}

    if unexpected_values:
        raise ValueError(
            f"Unexpected values found in {column}: "
            f"{unexpected_values}"
        )

    binary_validation.append({
        "Column": column,
        "Unique Values": sorted(unique_values),
        "Missing Values": int(
            df_clean[column].isna().sum()
        )
    })

binary_validation_summary = pd.DataFrame(binary_validation)

binary_validation_summary

,Column,Unique Values,Missing Values
0,current_smoker,"[0, 1]",0
1,bp_meds,"[0, 1]",53
2,prevalent_stroke,"[0, 1]",0
3,prevalent_hypertension,"[0, 1]",0
4,diabetes,"[0, 1]",0
